In [44]:
# Cell 1: Setup
%load_ext autoreload
%autoreload 2
from datetime import datetime
import audit_engine as ae
import pandas as pd
pd.set_option('display.max_columns', None)
import time
import concurrent.futures
from tqdm.auto import tqdm
import os
from datetime import date,timedelta
import duckdb

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [48]:
import importlib
import audit_engine as ae

# 1. Reload the specific sub-module where the logic changed
importlib.reload(ae.fs_api)
importlib.reload(ae.config)
importlib.reload(ae.core)
importlib.reload(ae.utils)
importlib.reload(ae.db_logic)
importlib.reload(ae)

# 2. Reload the main package to refresh the __init__ mappings
importlib.reload(ae)

print("all libs reimported and ae namespace updated.")

import gc
import pandas as pd

# This clears up unreferenced objects/file handles
gc.collect()

all libs reimported and ae namespace updated.


1652

In [3]:
ts_prefix = datetime.now().strftime("%Y%m%d%H%M")
fn = f'data_reports/tickets_{ts_prefix}.json'
conn = ae.get_sf_connection()

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/ef6f731e-41f8-43bd-9df6-8982a75c2f24/saml2?SAMLRequest=lZJRb9owGEX%2FSuQ9J7FNgMQCKihCQ4KOlXTS%2BmYSB7w6NvhzmvLvZ0KRuodW2lvknGsf%2B36ju7daBa%2FCgjR6jEiEUSB0YUqp92P0lC%2FCFAXguC65MlqM0VkAupuMgNfqyKaNO%2BhHcWoEuMBvpIF1P8aosZoZDhKY5rUA5gq2na5XjEaYHa1xpjAKfYh8neAAwjpveIuUIL3ewbkji%2BO2baO2Fxm7jynGOMZZ7KkL8u3Gv%2Fk7fcKTGCcX3hMe37y7zaS%2BPsFXWrsrBOx7nm%2FCzY9tjoLpTfXeaGhqYbfCvspCPD2urgLgDeozJ4MsIVEDoeDgQhKBNm2l%2BIsoTH1snN828l9xJcpYmb30N1%2FOx%2Bj4IsvioaVqI3c%2F29kgnR3yckW3yz%2BL34avs8V%2B%2BHCy7jRb7TP1vC5Q8OtWLb1UuwRoxFJfCnV%2BCdNBiEnYIzmmLMGs349wSp9RMPeFSs1dl7xZdx5RLQtrwFTOaCW16CxFNaiGPSLChFRpmPR2ZZiV1SBMs5TyYb%2BgFU3iS80UXUeHdSJ28t8PMoo%2Fxt%2FH8ME3s5xvjJLFOVgYW3P3eXEkIt2KLMOqQ5mouVTTsrQCwBeolGnvreDOT7uzjUDx5Hrqv%2FM%2B%2BQs%3D&RelayState=ver%3A1-hint%3A27309133378650118-ETMsDgAAAZwR7MMqABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEFxNWh2rI8%2FvN8b4DUJYd5IAAA

In [5]:
print(f"Started at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 1. Fetch tickets using the engine
# Note: Ensure start_date and end_date are valid strings for normalize_date
# 1. Get Input
raw_input = input('Enter date (yyyy-mm-dd) or days back (max 60): ').strip()

# 2. Define our Safety Limit
MAX_LOOKBACK = 60
default_date = date.today() - timedelta(days=20)

# 3. Logic Flow
if not raw_input:
    final_date = default_date

elif raw_input.isdigit():
    days = int(raw_input)
    # Cap the days at 60
    days = min(days, MAX_LOOKBACK)
    final_date = date.today() - timedelta(days=days)

else:
    try:
        # Try to parse the specific date string
        parsed_date = datetime.strptime(raw_input, '%Y-%m-%d').date()
        
        # Check if the entered date is older than 60 days
        earliest_allowed = date.today() - timedelta(days=MAX_LOOKBACK)
        
        if parsed_date < earliest_allowed:
            print(f"Date too old! Capping at {MAX_LOOKBACK} days ago.")
            final_date = earliest_allowed
        else:
            final_date = parsed_date
            
    except ValueError:
        # This catches "abcde" or "2023-13-45"
        print("Invalid input detected. Defaulting to 20 days ago.")
        final_date = default_date

sd1 = final_date.strftime('%Y-%m-%d')
print(f"Final Start Date: {sd1}")

ed1 = input(prompt='enter end date in format yyyy-mm-dd : ').strip()
if not ed1:
    ed1 = date.today().strftime('%Y-%m-%d')
print(f"Start Date set to: {sd1}")
print(f"End Date set to: {ed1}")

sd2 = sd1+'T00:00:00Z'
ed2 = ed1+'T00:00:00Z'

# tickets = ae.get_tickets_between_dates(
#     start_date = "2026-01-15T00:00:00Z",
#     end_date = "2026-01-27T23:59:59Z"
# )
tickets = ae.get_tickets_between_dates(
    start_date = sd2,
    end_date = ed2
)

if 1==1:
    ae.fs_api.save_or_load_tickets(
        tickets=tickets,
        ts_prefix=ts_prefix,
        mode="save"
    )
    
    # 3. Load tickets back for verification
    tickets = ae.fs_api.save_or_load_tickets(
        filename=fn,
        mode="load"
    )
print(f"\n Total tickets loaded: {len(tickets)}")
print(f"Ended at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Started at 2026-01-30 21:21:18


KeyboardInterrupt: Interrupted by user

In [6]:
# --- YOUR CONTROLS START ---
time.sleep(65)
target_tickets = []

for t in tickets:
    ticko_subject = t.get('subject', '').lower().replace(' ', '')
    cf = t.get('custom_fields', {})
    audit_status = cf.get('audit_status_1')
    cleaned_subject = ticko_subject.lower().replace(' ', '')
    is_price_change = ("pricechange" in cleaned_subject and "external" not in cleaned_subject)
    audit_status_check = bool(audit_status and audit_status.strip().lower() == 'new')
    audit_status_check = True
    
    if is_price_change and audit_status and audit_status_check:
        target_tickets.append(t)
# --- YOUR CONTROLS END ---

print(f"Filtered down to {len(target_tickets)} tickets. Starting parallel run...")

# 1. Start Timing
start_time = datetime.now()

# 2. Parallel Extraction (Using the list you just built)
price_change_df = ae.process_tickets_parallel(target_tickets, max_workers=10)
price_change_df = ae.build_price_change_df(df_rows=price_change_df.to_dict('records'), ts_prefix=ts_prefix,save_excel=False)

# 3. End Timing
print(f"Total Duration: {datetime.now() - start_time}")

Filtered down to 381 tickets. Starting parallel run...
Starting parallel extraction for 381 tickets...


API Extraction:   0%|          | 0/381 [00:00<?, ?it/s]


[Rate Limit] 120 calls reached. Sleeping 65s...
[Rate Limit] 240 calls reached. Sleeping 65s...
Total Duration: 0:03:48.142506. Sleeping 65s...


In [7]:
# Calculate counts and basic checks
price_change_df['ticko_pcw_worksheet_details_ct'] = price_change_df['ticko_pcw_worksheet_details'].apply(len)
# Validation: Ensure details count is numeric
price_change_df['worksheet_checks'] = pd.to_numeric(price_change_df["ticko_pcw_worksheet_details_ct"], errors="coerce").notnull()

print('df_rows length: ', len(price_change_df.to_dict('records')))
print('price_change_df shape: ', price_change_df.shape)

# 4. Reorder and Rename using Config
# We pull 'price_change_df_col_order' and 'price_change_df_rename_dict' from ae.config
price_change_df_send = ae.utils.reorder_and_rename_columns(
    price_change_df, 
    ae.config.price_change_df_col_order, 
    ae.config.price_change_df_rename_dict
)

# 5. Business Logic Formatting
price_change_df_send['attachment_ready'] = 0
price_change_df_send['old_unique_key_list'] = 0

# Map Markdowns using your config dict
price_change_df_send['Markdown_Code'] = (
    price_change_df_send['Markdown']
    .map(ae.config.mkdown_dict)
    .fillna(99)
    .astype('Int64')
)

# Fix Dates
price_change_df_send['Effective_Date_fixed'] = pd.to_datetime(price_change_df_send['Effective_Date2']).dt.date
price_change_df_send['Ticket_Creation_DT'] = pd.to_datetime(price_change_df_send['Ticket_Creation_Datetime']).dt.date
price_change_df_send['Worksheet_Details'] = price_change_df_send['Worksheet_Details'].astype(str)
price_change_df_send['Percent_Off2'] = price_change_df_send['Percent_Off'].astype(str).str.extract(r'(\d+)')[0].astype(float)
price_change_df_send['Percent_Off2'] = price_change_df_send['Percent_Off2'].apply(ae.utils.normalize_pct)

# 6. Save the Audit Draft
output_fn = f'data_reports/price_change_df_send_1_{ts_prefix}.xlsx'
price_change_df_send.to_excel(output_fn, index=False)
print(f"Audit file saved: {output_fn}")

df_rows length:  489
price_change_df shape:  (489, 33)
Audit file saved: price_change_df_send_1_202601302119.xlsx


In [8]:
time.sleep(65)
ae.utils.delete_all_files(ae.config.down_path)

break_limit = 5000

print(f"Starting downloads for {len(price_change_df_send)} potential attachments...")
# Prepare tasks from the dataframe
tasks = [
    (row["canonical_last_part"], row["unique_key"]) 
    for _, row in price_change_df_send.iterrows()
    if pd.notna(row["canonical_last_part"]) and len(str(row["canonical_last_part"])) > 8
]

# Run the bulk download
ae.fs_api.bulk_download_attachments_v2(tasks, max_workers=5)

print(f"\nFinished. Files are in: {ae.config.down_path}")

Starting downloads for 489 potential attachments...
Initiating parallel download with 5 workers...


[autoreload of audit_engine.fs_api failed: Traceback (most recent call last):
  File "C:\Users\MBathija\AppData\Local\anaconda3\envs\freshaudit\Lib\site-packages\IPython\extensions\autoreload.py", line 322, in check
    elif self.deduper_reloader.maybe_reload_module(m):
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\MBathija\AppData\Local\anaconda3\envs\freshaudit\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 545, in maybe_reload_module
    new_source_code = f.read()
                      ^^^^^^^^
  File "C:\Users\MBathija\AppData\Local\anaconda3\envs\freshaudit\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 522: character maps to <undefined>
]


Downloading:   0%|          | 0/489 [00:00<?, ?it/s]


[Rate Limit] Reached 100 requests. Sleeping for 65s...
[Rate Limit] Reached 200 requests. Sleeping for 65s...
[Rate Limit] Reached 300 requests. Sleeping for 65s...
[Rate Limit] Reached 400 requests. Sleeping for 65s...
Finished. Files are in: C:\Users\MBathija\BI_Engineering\FreshAuditv2\Down_Folder


In [9]:
ae.utils.delete_all_files("Fixed_attachments")

attachment_df_list = []
issue_df_list = []
break_limit = 5000

# 2. Main Processing Loop
for enm, (idx, can_id, uk, tkt, tkt_dt) in enumerate(zip(
    price_change_df_send.index,
    price_change_df_send["canonical_last_part"], 
    price_change_df_send["unique_key"],
    price_change_df_send['Ticket_Id'],
    price_change_df_send['Ticket_Creation_DT']
)):
    
    # Pre-filtering logic
    cid = str(can_id).strip()
    if pd.isna(can_id) or cid.lower() == 'nan' or len(cid) <= 5:
        continue
    if enm >= break_limit:
        break
        
    # Construct paths using ae.config
    file_path = os.path.join(ae.config.down_path, uk if uk.lower().endswith(".xlsx") else f"{uk}.xlsx")
    
    if not os.path.exists(file_path):
        continue

    try:
        # Use the reader from your core logic
        # Assuming COL_MAP and REQUIRED_SETS are now in ae.config or ae.core
        temp1, missing_cols = ae.core.read_price_change_sheet_v4(
            file_path, 
            ae.config.COL_MAP, 
            ae.config.REQUIRED_SETS
        )
        
        print(f"Processing: {enm}", end="\r")

        if temp1 is None:
            issue_df_list.append([uk, tkt, tkt_dt, file_path, f"Schema mismatch: {missing_cols}"])
            continue

        # Data Cleaning
        temp1 = temp1.replace(r'^\s*$', pd.NA, regex=True).dropna(how="all").reset_index(drop=True)
        
        # Remove "Unnamed" columns
        unnamed = [c for c in temp1.columns if str(c).lower().startswith("unnamed")]
        temp1 = temp1.drop(columns=unnamed, errors="ignore")
        temp1 = temp1.loc[:, temp1.columns.notna() & (temp1.columns.astype(str).str.strip() != "")]

        # SKU Logic using SKU_COL_SETS from config
        unique_skus = None
        for cols in ae.config.SKU_COL_SETS:
            if all(c in temp1.columns for c in cols):
                unique_skus = temp1[cols].drop_duplicates().shape[0]
                break
        
        temp1['unique_skus'] = unique_skus
        temp1['uk'] = uk
        attachment_df_list.append(temp1)

        # Save to "Fixed" directory using path from config
        fixed_path = os.path.join(ae.config.attachment_path, f"fixed_{os.path.basename(file_path)}")
        temp1.to_excel(fixed_path, index=False)
        
        # Update your main tracking DF
        price_change_df_send.loc[idx, 'attachment_ready'] = 1

    except Exception as e:
        issue_df_list.append([uk, tkt, tkt_dt, file_path, str(e)])
        continue

print(f"\nFinished. Successes: {len(attachment_df_list)} | Issues: {len(issue_df_list)}")

Processing: 292

C:\Users\MBathija\AppData\Local\anaconda3\envs\freshaudit\Lib\site-packages\openpyxl\reader\workbook.py:84: UserWarning: File contains an invalid specification for 0. This will be removed
  warn(msg)


Processing: 339

C:\Users\MBathija\AppData\Local\anaconda3\envs\freshaudit\Lib\site-packages\openpyxl\reader\workbook.py:84: UserWarning: File contains an invalid specification for 0. This will be removed
  warn(msg)


Processing: 488
Finished. Successes: 224 | Issues: 264


In [10]:
issue_files = pd.DataFrame(issue_df_list,columns=['file_id','ticket','ticket_create_date','path','error'])
print('issue templates are : '+str(issue_files.shape[0]))
print('correct templates are : '+str(price_change_df_send[price_change_df_send['attachment_ready']==1].shape[0]))
print('issue rate is : '+str(round(issue_files.shape[0]/price_change_df_send.shape[0],2)*100)+'%')
issue_files['size_kb'] = issue_files['path'].apply(lambda x: round(os.path.getsize(x) / 1024, 2) if os.path.exists(x) else 0)
issue_files['missing_count'] = issue_files['error'].apply(ae.utils.get_count)
issue_files.sort_values(by=['missing_count'],inplace=True)

issue templates are : 264
correct templates are : 224
issue rate is : 54.0%


In [11]:
# 1. First loop: Merge attachments
dub_can_tickets = price_change_df_send[(price_change_df_send['canonical_url_ct'] > 1) & (price_change_df_send['attachment_ready'] == 1)]['Ticket_Id'].unique().tolist()

if len(dub_can_tickets) > 0:
    print(dub_can_tickets)
    for i in dub_can_tickets:
        # 1. Reset the list for every new ticket
        current_ticket_dfs = []
        
        # Get UKs for this specific ticket
        mask = (price_change_df_send['Ticket_Id'] == i) & (price_change_df_send['attachment_ready'] == 1)
        uks = price_change_df_send[mask]['unique_key'].unique().tolist()
        
        if len(uks) > 1:
            for uk in uks:
                # Pointing to your package config path
                fixed_path = os.path.join(ae.config.attachment_path, f"fixed_{uk}.xlsx")
                
                if os.path.exists(fixed_path):
                    attach = pd.read_excel(fixed_path)
                    attach['orig_file'] = uk
                    current_ticket_dfs.append(attach)
    
            # 2. Combine and handle the result for this specific ticket
            if current_ticket_dfs:
                combined_for_i = pd.concat(current_ticket_dfs, ignore_index=True)
                # Note: uk here will be the last one from the loop above
                merged_filename = f"fixed_{uk}_merged.xlsx"
                new_path = os.path.join(ae.config.attachment_path, merged_filename)
                
                combined_for_i.to_excel(new_path, index=False) # index=False usually cleaner for Excel
                temp_name = merged_filename.replace('.xlsx','').replace('fixed_','')
                price_change_df_send.loc[price_change_df_send['Ticket_Id'] == i, 'unique_key'] = temp_name
                uks_string = ae.utils.list_to_str(uks)
                price_change_df_send.loc[price_change_df_send['Ticket_Id'] == i, 'old_unique_key_list'] = uks_string

# 2. Second loop: Deduplicate and format
if len(dub_can_tickets) > 0:
    for i in dub_can_tickets:
        ignore_cols = ['tickt_canonical_url', 'canonical_last_part','old_unique_key_list'] 
        cols_to_use = [c for c in price_change_df_send.columns if c not in ignore_cols]
        
        price_change_df_send['Worksheet_Details'] = price_change_df_send['Worksheet_Details'].astype(str)
        price_change_df_send = price_change_df_send.drop_duplicates(subset=cols_to_use).reset_index(drop=True)
        
        # Using the utility from your package
        price_change_df_send['Worksheet_Details'] = price_change_df_send['Worksheet_Details'].apply(ae.utils.string_to_list_quoted)

# 3. Final Export
price_change_df_send['Worksheet_Details'] = price_change_df_send['Worksheet_Details'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
price_change_df_send.to_excel(f'data_reports/price_change_df_send_2_{ts_prefix}.xlsx', index=False)

[119286, 119267, 118938, 118934, 118867, 118741, 118625, 118575, 118492, 117608, 117478, 117474, 117442, 117340]


C:\Users\MBathija\AppData\Local\Temp\ipykernel_5444\1768892972.py:35: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '202601302119_119286_27083537016_00000033, 202601302119_119286_27083537017_00000034' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  price_change_df_send.loc[price_change_df_send['Ticket_Id'] == i, 'old_unique_key_list'] = uks_string


In [159]:
audit_df = []
ae.utils.delete_all_files("SQL_Files")
print(f"Starting Audit Loop for {len(price_change_df_send)} rows...")

for enm, (_, r) in enumerate(price_change_df_send.iterrows()):
    Price_Change_Type_var = str(r['Price_Change_Type']).strip().lower() if r['Price_Change_Type'] is not None else ""
    Price_Change_Type_check = Price_Change_Type_var in ['perm markdown','regular / regular','markdown cancel']
    attachment_ready_check = r['attachment_ready'] == 1
    single_ticket_chk = True
    single_ticket_chk = r['Ticket_Id']== 119628
    if not (Price_Change_Type_check and attachment_ready_check and single_ticket_chk):
        continue

    # --- Print Progress ---
    msg = f"[{enm}] Processing Ticket: {r['Ticket_Id']} | UK: {r['unique_key']}"
    print(f"\r{msg.ljust(150)}", end='')

    # 2. Initial Variable Mapping
    ts_orig = pd.to_datetime(r['Ticket_Creation_Datetime'])
    fs_date = pd.to_datetime(audit_dict['fs_effective_date']).date()
    audit_dict = {
        'fs_uk': r['unique_key'],
        'fs_ticketid': r['Ticket_Id'],
        'fs_audit_status':r['Audit_Status'],
        'fs_Cost_Updates' : r['Cost_Updates'],
        'fs_markdown': r['Markdown_Code'],
        'fs_Price_Change_Type': r['Price_Change_Type'],
        'fs_effective_date': fs_date,
        'fs_ticket_date': ts_orig.strftime('%Y-%m-%d'),
        'fs_ticket_ts': ts_orig.strftime('%Y-%m-%d %H:%M:%S'),
        'fs_wk_string': ae.utils.list_to_quoted_str_v2(r['Worksheet_Details']),
        'fs_perc_off':r['Percent_Off2'],
        'sql_shape': 0,
        'sql_ws_status_match': False,
        'chk_effective_date_match': False,
        'chk_markdown_code_match': False,
        'chk_fs_count_match': False,
        'chk_perc_match_flag': False,
        'chk_missing_summary_text': "No mismatch data"
    }
    fs_markdown_val = pd.to_numeric([r['Markdown_Code']], errors='coerce')[0] if pd.notnull(r['Markdown_Code']) else 0
    if len(audit_dict['fs_wk_string']) <= 4:
        audit_df.append(audit_dict)
        continue
    # 3. File Processing
    fixed_path = os.path.join(ae.config.attachment_path, f"fixed_{audit_dict['fs_uk']}.xlsx")
    sql_fixed_path = os.path.join(ae.config.sql_path, f"fixed_{audit_dict['fs_uk']}_sql.xlsx")
    
    if not os.path.exists(fixed_path):
        print(f"  ⚠️  File not found: {fixed_path}")
        audit_df.append(audit_dict)
        continue
        
    attach = pd.read_excel(fixed_path)
    attach = attach[~attach['FASHION_STYLE_NUMBER'].astype(str).str.contains(r'^9+(?:\.0+)?$', na=False)].reset_index(drop=True)
    attach = attach.dropna(thresh=2).reset_index(drop=True) # Will delete row if <2 columns have info/values
    attach['SELF_PROD_ID2'] = attach.apply(ae.core.build_conc_id, axis=1)
    attach['DEPARTMENT_NUMBER'] = pd.to_numeric(attach['DEPARTMENT_NUMBER'], errors='coerce').fillna(0).astype(int)
    if bool(attach['CHANGE_PERC'].isna().all()) & pd.notna(r['Percent_Off2']):
        attach['CHANGE_PERC'] = r['Percent_Off2']
    # attach_perc_list = ae.utils.list_to_str(attach['CHANGE_PERC'].apply(ae.utils.normalize_pct).unique().tolist())
    attach_perc_list = ae.utils.list_to_str(attach['CHANGE_PERC'].apply(ae.utils.normalize_pct).round(2).unique().tolist())
    # Calculate attachment metrics
    if Price_Change_Type_var in ['regular / regular','markdown cancel']:
        subset_cols = ['DEPARTMENT_NUMBER', 'MANUFACTURER_NUMBER', 'SELF_PROD_ID2']
    else:
        subset_cols = ['DEPARTMENT_NUMBER', 'MANUFACTURER_NUMBER', 'FASHION_STYLE_NUMBER']
    audit_dict['attach_fhs_count'] = len(attach.drop_duplicates(subset=subset_cols))
    audit_dict['attach_shape'] = attach.shape[0]

    audit_dict['attach_perc_list'] = attach_perc_list

    # 4. SQL Logic
    sql_queryf = ae.config.sql_query_1.replace('{wd_string}', audit_dict['fs_wk_string'])
    audit_dict['sql_query_run'] = sql_queryf
    try:
        if os.path.exists(sql_fixed_path):
            sql_df = pd.read_excel(sql_fixed_path)
        else:
            sql_df = ae.run_query(conn, sql_queryf)
            if not sql_df.empty:
                sql_df.to_excel(sql_fixed_path,index=False)
                time.sleep(1)
    except:
        print(f"SQL build issue for {r['Ticket_Id']}")
        audit_df.append(audit_dict)
        continue
    if not sql_df.empty:
        # Bulk update SQL metrics
        mfg_ws_dict = (sql_df[['MANUFACTURER_NUMBER', 'WORKSHEET_NO']].dropna().drop_duplicates().groupby('MANUFACTURER_NUMBER')['WORKSHEET_NO'].apply(lambda x: ", ".join(x.astype(int).astype(str))).to_dict())
        sql_perc_list = ae.utils.list_to_str(sql_df['CHANGE_PCT'].apply(ae.utils.normalize_pct).unique().tolist())
        audit_dict.update({
            'sql_shape': sql_df.shape[0],
            'sql_fhs_count': len(sql_df.drop_duplicates(subset=['FASHION_STYLE_NUMBER'])),
            'sql_effective_date': sql_df['EFFECTIVE_DATE'].max(),
            'sql_ws_status_list': sql_df['STATUS'].unique().tolist(),
            'sql_ws_status_match': sql_df[sql_df['STATUS'].str.upper().str.strip().isin(['PROC','INC', 'VOID', 'FAIL'])].shape[0]==0,
            'sql_price_type_code':sql_df['PRICE_TYPE_CODE'].unique().tolist(),
            'sql_special_instructions':sql_df['SPECIAL_INSTRUCTIONS'].unique().tolist(),
            'sql_special_inst_name':ae.utils.list_to_str(list(set([str(i).split(' ')[-1] for i in sql_df['SPECIAL_INSTRUCTIONS'].unique() if pd.notna(i)]))),
            'sql_req_reason_no':sql_df['REQ_REASON_NO'].unique().tolist(),
            'sql_perc_list':sql_perc_list
        })
        mask = sql_df['STATUS'].str.upper().str.strip().isin(['PROC','INC', 'VOID', 'FAIL'])
        ae.summary_txt_builder(mask,sql_df,'chk_ws_status_mismatch_summary_text',mfg_ws_dict,audit_dict)
        unique_types = audit_dict['sql_price_type_code']
        if r['Markdown_Code'] in ae.config.markdown_list12:
            mask = sql_df['PRICE_TYPE_CODE']!='R'
            ae.summary_txt_builder(mask,sql_df,'chk_pricetype_mismatch_summary_text',mfg_ws_dict,audit_dict)
            audit_dict['chk_price_type_code_match'] = (unique_types == ['R'])
        elif r['Markdown_Code'] in ae.config.regular_pcw:
            audit_dict['chk_price_type_code_match'] = (unique_types == ['Z'])
            mask = sql_df['PRICE_TYPE_CODE']!='Z'
            ae.summary_txt_builder(mask,sql_df,'chk_pricetype_mismatch_summary_text',mfg_ws_dict,audit_dict)
        else:
            audit_dict['chk_price_type_code_match'] = True
            
        if pd.to_numeric(r['Markdown_Code'], errors='coerce') == 24:
            sql_df['PRICE_DIFF'] = round((sql_df['CURRENT_TICKET_DOL']-sql_df['NEW_TICKET_DOL']),2).astype(float)
            top_3 = round(sql_df['PRICE_DIFF'], 2).value_counts().head(3)
            long_string = "Top 3 Price Diffs -> " + ", ".join([f"{v}: {c}" for v, c in top_3.items()])
            audit_dict['chk_penny_makrdown_1cent'] = long_string
            audit_dict['chk_price_diff_match_24md'] = bool((sql_df['PRICE_DIFF']==0.01).all())
        
        # Matches
        sql_date = pd.to_datetime(audit_dict['sql_effective_date']).date()
        fs_date = pd.to_datetime(audit_dict['fs_effective_date']).date()
        audit_dict['chk_effective_date_match'] = sql_date == fs_date
        mask = ~(pd.to_datetime(sql_df['EFFECTIVE_DATE']).dt.date == fs_date)
        ae.summary_txt_builder(mask,sql_df,'chk_effective_date_mismatch_summary_text',mfg_ws_dict,audit_dict)
        
        audit_dict['chk_markdown_code_match'] = bool((sql_df['REQ_REASON_NO'].astype(int) == fs_markdown_val).all())
        mask = ~(sql_df['REQ_REASON_NO'] == r['Markdown_Code'])
        ae.summary_txt_builder(mask,sql_df,'chk_markdown_mismatch_summary_text',mfg_ws_dict,audit_dict)
        # audit_dict['chk_markdown_code_match'] = bool((sql_df['REQ_REASON_NO'].astype(int) == int(audit_dict['fs_markdown'])).all())
        audit_dict['chk_fs_count_match'] = audit_dict['attach_fhs_count'] == audit_dict['sql_fhs_count']

        # 5. Percentage Validation (The Merge Logic)
        perc_chk = attach['CHANGE_PERC'].notna().any()
        pric_chk =  attach['SELF_PROD_ID2'].notna().any()
        if (perc_chk and fs_markdown_val!=21)|(pric_chk and fs_markdown_val==21):
            # Perform the merge once
            attach['_temp_key'] = ae.core.get_clean_key(attach)
            sql_df['_temp_key'] = ae.core.get_clean_key(sql_df)
            if Price_Change_Type_var in ['regular / regular','markdown cancel']:
                pct_df = duckdb.query(ae.config.sql_merge).to_df()
            else:
                pct_df = attach.merge(sql_df[['_temp_key', 'CHANGE_PCT', 'WORKSHEET_NO','CURRENT_TICKET_DOL','NEW_TICKET_DOL']], on='_temp_key', how='left')
            
            pct_df.drop(columns=['_temp_key'], inplace=True)
            attach.drop(columns=['_temp_key'], inplace=True)
            sql_df.drop(columns=['_temp_key'], inplace=True)

            # Normalize and Count
            for col in ['CHANGE_PCT', 'CHANGE_PERC']:
                pct_df[col] = pct_df[col].apply(ae.utils.normalize_pct)

            mask = None
            if Price_Change_Type_var == 'markdown cancel':
                audit_dict['chk_perc_match_flag'] = True
                audit_dict['chk_perc_match_ct'] = 0
                mask = pct_df['NEW_TICKET_DOL'].isna()
                ae.summary_txt_builder(mask,pct_df,'chk_missing_summary_text',mfg_ws_dict,audit_dict,rebuild_ws_dict=False)
                if (bool(pct_df['NEW_RETAIL_PRICE'].isna().all())):
                    pct_df['PRICE_MATCH'] = (pct_df['ORIGINAL_TKT_DOL'].astype(float).round(2)==pct_df['NEW_TICKET_DOL'].astype(float).round(2))
                    audit_dict['chk_reg_price_match'] = int(pct_df['PRICE_MATCH'].sum())
                    audit_dict['chk_reg_price_mismatch'] = int((~pct_df['PRICE_MATCH']).sum())
                    mask = ~pct_df['PRICE_MATCH'] & ~pct_df['WORKSHEET_NO'].isna()
                    ae.summary_txt_builder(mask,pct_df,'chk_reg_price_mismatch_summary_text',mfg_ws_dict,audit_dict)
                    audit_dict['chk_reg_price_flag'] = int((~pct_df['PRICE_MATCH']).sum())==0
                else:
                    audit_dict['chk_reg_price_match'] = int(pct_df['PRICE_MATCH'].sum())
                    audit_dict['chk_reg_price_mismatch'] = int((~pct_df['PRICE_MATCH']).sum())
                    audit_dict['chk_reg_price_flag'] = int((~pct_df['PRICE_MATCH']).sum())==0
                    mask = ~pct_df['PRICE_MATCH'] & ~pct_df['WORKSHEET_NO'].isna()
                    ae.summary_txt_builder(mask,pct_df,'chk_reg_price_mismatch_summary_text',mfg_ws_dict,audit_dict)
            elif Price_Change_Type_var == 'regular / regular':
                audit_dict['chk_perc_match_flag'] = True
                audit_dict['chk_perc_mismatch_ct'] = 0
                audit_dict['chk_perc_match_ct'] = 0
                audit_dict['chk_reg_price_match'] = int(pct_df['PRICE_MATCH'].sum())
                audit_dict['chk_reg_price_mismatch'] = int((~pct_df['PRICE_MATCH']).sum())
                audit_dict['chk_reg_price_flag'] = int((~pct_df['PRICE_MATCH']).sum())==0
                mask = pct_df['NEW_TICKET_DOL'].isna()
                ae.summary_txt_builder(mask,pct_df,'chk_missing_summary_text',mfg_ws_dict,audit_dict,rebuild_ws_dict=False)
                mask = ~pct_df['PRICE_MATCH'] & ~pct_df['WORKSHEET_NO'].isna()
                ae.summary_txt_builder(mask,pct_df,'chk_reg_price_mismatch_summary_text',mfg_ws_dict,audit_dict)
            elif Price_Change_Type_var == 'perm markdown':
                pct_df['PERC_MATCH'] = pct_df['CHANGE_PERC'] == pct_df['CHANGE_PCT']
                audit_dict['chk_reg_price_match'] = 0
                audit_dict['chk_reg_price_mismatch'] = 0
                audit_dict['chk_reg_price_flag'] = True
                audit_dict['chk_perc_mismatch_ct'] = int((pct_df['PERC_MATCH'] == False).sum()),
                audit_dict['chk_perc_match_ct'] = int((pct_df['PERC_MATCH'] == True).sum()),
                audit_dict['chk_perc_match_flag'] = bool(pct_df[pct_df['CHANGE_PCT'].notna()]['PERC_MATCH'].all())
                mask = pct_df['CHANGE_PCT'].isna()
                ae.summary_txt_builder(mask,pct_df,'chk_missing_summary_text',mfg_ws_dict,audit_dict,rebuild_ws_dict=False)
                mask = (pct_df['PERC_MATCH'] == False) & (~pct_df['CHANGE_PCT'].isna())
                ae.summary_txt_builder(mask,pct_df,'chk_incorrect_perc_summary_text',mfg_ws_dict,audit_dict)
            
            # Update Audit with Comparison Results
            audit_dict.update({
                'attach_pctdf_shape': pct_df.shape[0],
                'chk_style_ct_mismatch': int(pct_df['CHANGE_PCT'].isna().sum()),
                'chk_style_ct_match': int(pct_df['CHANGE_PCT'].notna().sum())
            })
        else:
            audit_df.append(audit_dict)
            continue
    else:
        pass
        print(f"❌ SQL Result Empty for Ticket {r['Ticket_Id']}",end="\r",flush=True)

    audit_df.append(audit_dict)

print(f"\nAudit complete. {len(audit_df)} records added to audit_df.")

Starting Audit Loop for 464 rows...
[0] Processing Ticket: 119628 | UK: 202601302119_119628_27083824452_00000001                                                                          
Audit complete. 1 records added to audit_df.


In [110]:
sql_full = ae.utils.merge_xlsx_files('SQL_Files',f'data_reports/SQL_Full_Export_{ts_prefix}.xlsx')
attach_full = ae.utils.merge_xlsx_files('Fixed_attachments',f'data_reports/Attachments_Full_Export_{ts_prefix}.xlsx')
attach_full['DEPARTMENT_NUMBER'] = attach_full['DEPARTMENT_NUMBER'].fillna(0).astype('int64',errors='ignore')
attach_full['MANUFACTURER_NUMBER'] = attach_full['MANUFACTURER_NUMBER'].fillna(0).astype('int64',errors='ignore')
attach_full['FASHION_STYLE_NUMBER'] = attach_full['FASHION_STYLE_NUMBER'].fillna(0).astype('int64',errors='ignore')
sql_full['ticket_id'] = (sql_full['source_file'].fillna('').str.replace('fixed_', '', case=False).str.split('_', n=2, expand=True)[1].fillna('error_no_id'))
attach_full['ticket_id'] = (attach_full['source_file'].fillna('').str.replace('fixed_', '', case=False).str.split('_', n=2, expand=True)[1].fillna('error_no_id'))
col = sql_full.pop('ticket_id')
sql_full.insert(0, 'ticket_id', col)
col = attach_full.pop('ticket_id')
attach_full.insert(0, 'ticket_id', col)
sql_full['_temp_key'] = ae.core.get_clean_key(sql_full)
attach_full['_temp_key'] = ae.core.get_clean_key(attach_full)

Found 175 files. Starting merge...
Found 237 files. Starting merge...


In [149]:
audit_df_final = pd.DataFrame(audit_df)
audit_df_final.loc[audit_df_final['fs_markdown'] == 21, ['chk_perc_match_flag']] = True
flag_cols = ['sql_ws_status_match','chk_effective_date_match','chk_markdown_code_match','chk_fs_count_match',
             'chk_perc_match_flag','chk_price_type_code_match','chk_reg_price_flag']
audit_df_final['chk_all_flags_match'] = audit_df_final[flag_cols].all(axis=1)
audit_df_final['chk_all_flags_match'] = audit_df_final['chk_all_flags_match'].fillna(False)
audit_df_final['flags_count_matched'] = audit_df_final[flag_cols].sum(axis=1)
audit_df_final['flags_count_failed'] = len(flag_cols) - audit_df_final['flags_count_matched']
audit_df_final['failed_reasons'] = audit_df_final[flag_cols].apply(lambda x: ' | '.join([col.replace('chk_', '').replace('_match', '').replace('_', ' ').strip().upper() for col in x.index[~x.astype(bool)]]) if not x.all() else 'PASS', axis=1)
audit_df_final['pass_reasons'] = audit_df_final[flag_cols].apply(lambda x: ' | '.join([col.replace('chk_', '').replace('_match', '').replace('_', ' ').strip().upper() for col in x.index[x.astype(bool)]]) if x.any() else 'NONE', axis=1)
audit_df_final['sql_query_run'] = audit_df_final['sql_query_run'].str.strip()
pcols = ae.config.audit_df_final_col_order
audit_df_final = audit_df_final[pcols + audit_df_final.columns.difference(pcols).tolist()]
audit_df_final[flag_cols] = audit_df_final[flag_cols].fillna(True)
wrap_targets = ['chk_missing_summary_text','sql_query_run'] # Add more column names to this list as needed
color_targets = [c for c in audit_df_final.columns if 'chk_' in c or 'match' in c]
# ---------------------------

# 2. Logic Functions
def apply_excel_styles(val):
    style = ''
    if val is False:
        style += 'background-color: #FFC7CE; color: #9C0006;'
    elif val is True:
        style += 'background-color: #C6EFCE; color: #006100;'
    return style

def force_wrap(val):
    return 'white-space: pre-wrap;'

# 3. Apply Styling via Pandas Styler
styled_audit_df = audit_df_final.style.applymap(
    apply_excel_styles, subset=color_targets
).applymap(
    force_wrap, subset=wrap_targets
)

# 4. Export Setup
output_fn = f"data_reports/Final_Audit_Results_{ts_prefix}.xlsx"
writer = pd.ExcelWriter(output_fn, engine='xlsxwriter')
styled_audit_df.to_excel(writer, index=False, sheet_name='Audit Summary')

workbook  = writer.book
worksheet = writer.sheets['Audit Summary']

# 5. Global Formatting
normal_format = workbook.add_format({'valign': 'vcenter'})
# Set all columns to standard width (1x)
worksheet.set_column(0, len(audit_df_final.columns) - 1, 15, normal_format)

# 6. Apply List-based Width and Height logic
# Set the wide width for everything in your wrap list
for col_name in wrap_targets:
    if col_name in audit_df_final.columns:
        idx = audit_df_final.columns.get_loc(col_name)
        worksheet.set_column(idx, idx, 50) # 3x-ish Width

# Set all rows to 3x height (60 units)
for row_num in range(1, len(audit_df_final) + 1):
    worksheet.set_row(row_num, 60)

# 6.5 Convert the data range into an Excel Table
(max_row, max_col) = audit_df_final.shape
column_settings = [{'header': column} for column in audit_df_final.columns]

worksheet.add_table(0, 0, max_row, max_col - 1, {
    'columns': column_settings,
    'style': 'Table Style Medium 9', # You can change the number for different colors
    'name': 'AuditTable'
})
# Add SQL Full Raw Data
if 'sql_full' in locals() and not sql_full.empty:
    sql_full.to_excel(writer, index=False, sheet_name='SQL Raw Data')

if 'attach_full' in locals() and not attach_full.empty:
    attach_full.to_excel(writer, index=False, sheet_name='Attach Raw Data')

if 'issue_files' in locals() and not issue_files.empty:
    issue_files.to_excel(writer, index=False, sheet_name='Issue Files')
# 7. Close and Save
writer.close()

print(f"\n" + "="*100)
print(f"✅ Report Fixed & Generated: {output_fn}")
print(f"📝 Wrapped Columns: {wrap_targets}")
print(f"📊 Colored Columns: {len(color_targets)} identified")
print("="*100)
print(f"Ended at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

C:\Users\MBathija\AppData\Local\Temp\ipykernel_43448\3718600566.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  audit_df_final[flag_cols] = audit_df_final[flag_cols].fillna(True)
C:\Users\MBathija\AppData\Local\Temp\ipykernel_43448\3718600566.py:32: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styled_audit_df = audit_df_final.style.applymap(
C:\Users\MBathija\AppData\Local\Temp\ipykernel_43448\3718600566.py:34: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  ).applymap(



✅ Report Fixed & Generated: data_reports/Final_Audit_Results_202601310811.xlsx
📝 Wrapped Columns: ['chk_missing_summary_text', 'sql_query_run']
📊 Colored Columns: 22 identified
Ended at 2026-01-31 16:50:35
